In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
def explore_dataset(root_dir):
    classes = {'shop lifters': 1, 'non shop lifters': 0}
    data = []
    
    for class_name, label in classes.items():
        class_dir = os.path.join(root_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"Directory not found: {class_dir}")
            continue
            
        video_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
        print(f"Class '{class_name}': Found {len(video_files)} videos.")
        
        for video_file in video_files:
            video_path = os.path.join(class_dir, video_file)
            cap = cv2.VideoCapture(video_path)
            
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            cap.release()
            
            duration = frame_count / fps if fps > 0 else 0
            
            data.append({
                'class': class_name,
                'name': video_file,
                'frames': frame_count,
                'fps': fps,
                'duration': duration,
                'resolution': f"{width}x{height}"
            })
            
    df = pd.DataFrame(data)
    
    if not df.empty:
        print("\n=== Dataset Summary Statistics ===")
        summary = df.groupby('class').agg({
            'name': 'count',
            'duration': ['mean', 'min', 'max'],
            'frames': ['mean', 'min', 'max']
        })
        print(summary)
        
        print("\nResolution Distribution:")
        print(df['resolution'].value_counts())
    else:
        print("No videos were processed.")
        
    return df

In [ ]:
class ShopliftingDataset(Dataset):
    def __init__(self, video_paths, labels, T=16, is_train=True):
        self.video_paths = video_paths
        self.labels = labels
        self.T = T
        self.is_train = is_train

    def _sample_frame_indices(self, total_frames):
        if total_frames >= self.T:
            indices = np.linspace(0, total_frames - 1, self.T, dtype=int)
        else:
            indices = np.linspace(0, total_frames - 1, total_frames, dtype=int)
            padding = [total_frames - 1] * (self.T - total_frames)
            indices = np.concatenate([indices, padding])
        return indices

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        cap = cv2.VideoCapture(video_path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()

        total_frames = len(frames)
        if total_frames == 0:
            return torch.zeros((self.T, 3, 112, 112)), label

        sampled_indices = self._sample_frame_indices(total_frames)
        sampled_frames = [frames[i] for i in sampled_indices]

        transformed_frames = []

        if self.is_train:
            crop_i = random.randint(0, 128 - 112)
            crop_j = random.randint(0, 128 - 112)
            do_flip = random.random() > 0.5
            
            brightness = random.uniform(0.8, 1.2)
            contrast = random.uniform(0.8, 1.2)
            saturation = random.uniform(0.8, 1.2)

            for f in sampled_frames:
                img = Image.fromarray(f)
                img = TF.resize(img, (128, 128))
                img = TF.crop(img, crop_i, crop_j, 112, 112)
                
                if do_flip:
                    img = TF.hflip(img)
                    
                img = TF.adjust_brightness(img, brightness)
                img = TF.adjust_contrast(img, contrast)
                img = TF.adjust_saturation(img, saturation)
                
                tensor_img = TF.to_tensor(img)
                tensor_img = TF.normalize(tensor_img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                transformed_frames.append(tensor_img)
        else:
            for f in sampled_frames:
                img = Image.fromarray(f)
                img = TF.resize(img, (128, 128))
                img = TF.center_crop(img, (112, 112))
                
                tensor_img = TF.to_tensor(img)
                tensor_img = TF.normalize(tensor_img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                transformed_frames.append(tensor_img)

        clip_tensor = torch.stack(transformed_frames)
        return clip_tensor, label

In [ ]:
class VideoClassifier3D(nn.Module):
    def __init__(self, num_classes=2):
        super(VideoClassifier3D, self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv3d(3, 16, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2)),
            
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2)),
            
            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2)),
            
            nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1, 3, 4)
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch_idx):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch_idx+1} [Train]", leave=True)
    
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
        # Predictions Tracking
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
        
    epoch_loss = running_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    
    return epoch_loss, acc, prec, rec, f1


def evaluate_model(model, dataloader, criterion, device, desc="Evaluating"):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(dataloader, desc=desc, leave=False)
    
    with torch.no_grad():
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
            
    epoch_loss = running_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    
    return epoch_loss, acc, prec, rec, f1

In [ ]:
def plot_metrics(history):
    """
    Plots Train vs Validation Loss, Accuracy, and F1-score side-by-side.
    """
    epochs = range(1, len(history['train_loss']) + 1)
    
    plt.figure(figsize=(18, 5))
    
    # Loss
    plt.subplot(1, 3, 1)
    plt.plot(epochs, history['train_loss'], 'bo-', label='Training Loss')
    plt.plot(epochs, history['val_loss'], 'ro-', label='Validation Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs, history['train_acc'], 'bo-', label='Training Acc')
    plt.plot(epochs, history['val_acc'], 'ro-', label='Validation Acc')
    plt.title('Training & Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # F1-Score
    plt.subplot(1, 3, 3)
    plt.plot(epochs, history['train_f1'], 'bo-', label='Training F1')
    plt.plot(epochs, history['val_f1'], 'ro-', label='Validation F1')
    plt.title('Training & Validation F1-Score')
    plt.xlabel('Epochs')
    plt.ylabel('F1-Score')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    # Save a static image copy to disk
    plt.savefig('training_validation_metrics.png', dpi=300)
    print("Metrics plot saved successfully as 'training_validation_metrics.png'")
    plt.show()

In [ ]:
def run_pipeline(root_dataset_dir, epochs=10, batch_size=8, target_frames=16):
    video_paths = []
    labels = []
    
    classes = {'non shop lifters': 0, 'shop lifters': 1}
    for class_name, label in classes.items():
        class_dir = os.path.join(root_dataset_dir, class_name)
        if not os.path.exists(class_dir):
            continue
        for f in os.listdir(class_dir):
            if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                video_paths.append(os.path.join(class_dir, f))
                labels.append(label)
                
    if len(video_paths) == 0:
        print("No videos found. Please check your dataset folder configuration.")
        return

    train_paths, test_paths, train_labels, test_labels = train_test_split(
        video_paths, labels, test_size=0.30, random_state=42, stratify=labels
    )
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        test_paths, test_labels, test_size=0.50, random_state=42, stratify=test_labels
    )
    
    print(f"Dataset Split Sizes -> Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")
    
    train_dataset = ShopliftingDataset(train_paths, train_labels, T=target_frames, is_train=True)
    val_dataset = ShopliftingDataset(val_paths, val_labels, T=target_frames, is_train=False)
    test_dataset = ShopliftingDataset(test_paths, test_labels, T=target_frames, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running execution on device: {device}\n")
    
    model = VideoClassifier3D(num_classes=2).to(device)
    
    # Class weights
    total_samples = 531 + 324
    weight_normal = total_samples / (2.0 * 531)
    weight_theft = total_samples / (2.0 * 324)
    class_weights = torch.FloatTensor([weight_normal, weight_theft]).to(device)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    
    best_val_loss = float('inf')
    save_path = "best_video_classifier.pth"
    
    # Structure to hold metric progress histories
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': []
    }
    
    for epoch in range(epochs):
        # Train and collect training evaluation metrics
        train_loss, train_acc, _, _, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, device, epoch
        )
        
        # Validate and collect validation metrics
        val_loss, val_acc, _, _, val_f1 = evaluate_model(
            model, val_loader, criterion, device, desc=f"Epoch {epoch+1} [Val]"
        )
        
        # Append parameters to the tracking log history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)
        
        print(f"Epoch {epoch+1}/{epochs} Summary | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} || "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} || "
              f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")
        
        # Checkpoint based on validation loss progression
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"--> Saved best model checkpoint to '{save_path}' (Val Loss: {val_loss:.4f})\n")
        else:
            print()
            
    # Draw visual metrics comparison 
    plot_metrics(history)
            
    # Final check on test dataset
    print("Loading the best model checkpoint for final test evaluation...")
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=device))
        
    test_loss, test_acc, test_prec, test_rec, test_f1 = evaluate_model(
        model, test_loader, criterion, device, desc="Testing"
    )
    
    print("\n=== Final Test Set Performance ===")
    print(f"Test Loss:  {test_loss:.4f}")
    print(f"Accuracy:   {test_acc:.4f}")
    print(f"Precision:  {test_prec:.4f}")
    print(f"Recall:     {test_rec:.4f}")
    print(f"F1-Score:   {test_f1:.4f}")

In [ ]:
dataset_directory = "/kaggle/input/datasets/omarabdelazez2004/shopdataset/Shop DataSet" 
    
explore_dataset(dataset_directory)
    
run_pipeline(dataset_directory, epochs=150, batch_size=4, target_frames=16)